# 03. Tool Use

**You will be able to:**

- Define tools the model can call
- Run the request/execute/respond loop with the SDK tool runner
- Handle tool errors without derailing the conversation

**Prerequisites:** Lesson 02  
**Estimated API cost:** ~$0.10

---

In [ ]:
import sys
sys.path.insert(0, "../src")

from genai_tutorial import ask, get_client, stream_text, usage_line, DEFAULT_MODEL

client = get_client()
print("Using model:", DEFAULT_MODEL)

## 1. A tool is a function plus a description

The docstring is not documentation for you — it is the prompt the model reads when deciding whether to call the tool. Vague docstrings produce wrong tool calls.

In [ ]:
from anthropic import beta_tool

@beta_tool
def get_weather(location: str, unit: str = "celsius") -> str:
    """Get the current weather for a location.

    Args:
        location: City and region, e.g. "Pune, MH".
        unit: Temperature unit, either "celsius" or "fahrenheit".
    """
    return f"22 degrees {unit} and clear in {location}"

## 2. The tool runner

The SDK drives the loop: call the API, run whatever tools the model asked for, feed results back, repeat until the model is done.

In [ ]:
runner = client.beta.messages.tool_runner(
    model=DEFAULT_MODEL,
    max_tokens=4000,
    tools=[get_weather],
    messages=[{"role": "user", "content": "Should I take a jacket in Pune today?"}],
)

for message in runner:
    for block in message.content:
        if block.type == "text":
            print(block.text)
        elif block.type == "tool_use":
            print(f"[calling {block.name} with {block.input}]")

## 3. What the runner is doing for you

Write the loop by hand once — it is about fifteen lines — so the abstraction stops being magic. Key rule: every `tool_use` block needs a matching `tool_result` with the same `tool_use_id`, and parallel calls all come back in a **single** user message.

In [ ]:
# TODO: implement the manual loop.
# while True: create -> if stop_reason != 'tool_use': break
#             -> execute each tool_use block -> append results -> repeat

## Exercise 03.1

Make `get_weather` raise for an unknown city. Return the failure as a `tool_result` with `is_error: True` and observe how the model recovers. Then try silently dropping the result instead — note how much worse that is.

In [ ]:
# Your code here

---

**Next:** see `README.md` for the full syllabus. Stuck? `docs/troubleshooting.md`. Solutions live in `solutions/` — try the exercise first.